# EAHN — Kaggle Execution Notebook
Run every cell **top to bottom** in order. Do not skip any cell.

| Cell | Purpose | Must pass before next? |
|------|---------|------------------------|
| 1 | Auto-detect dataset path + config | ✅ Yes |
| 2 | Sanity-check file counts | ✅ Yes |
| 3 | Clean up previous run | — |
| 4 | Clone repo from GitHub | ✅ Yes |
| 5 | Install dependencies | ✅ Yes |
| 6 | **Dataset loader verification** | ✅ Yes — must print PASSED |
| 7 | Train + Evaluate | — |
| 8 | Check output files | — |
| 9 | Display metrics table | — |
| 10 | Display detection graphs | — |
| 11 | Display heatmap videos | — |
| 12 | Print plain-English explanations | — |
| 13 | Full dashboard | — |
| 14 | Zip outputs for download | — |


---
## Cell 1 — Auto-detect dataset path + configuration

In [ ]:
import os, glob

# ── Auto-detect FF++ data root ────────────────────────────────────────────────
def find_ffpp_root(search_base="/kaggle/input", max_depth=5):
    candidates = []
    for root, dirs, _ in os.walk(search_base):
        depth = root.replace(search_base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        has_orig  = os.path.isdir(os.path.join(root, "original_sequences"))
        has_manip = os.path.isdir(os.path.join(root, "manipulated_sequences"))
        if has_orig and has_manip:
            candidates.append(root)
    return candidates

print("Scanning /kaggle/input for FF++ directory structure...")
found = find_ffpp_root()

if found:
    AUTO_ROOT = found[0]
    print(f"  Found: {AUTO_ROOT}")
    if len(found) > 1:
        print(f"  (Also found: {found[1:]} — using first match)")
else:
    AUTO_ROOT = None
    print("  NOT FOUND. See listing below and set DATA_ROOT manually.")

print("\n/kaggle/input layout (2 levels):")
for entry in sorted(glob.glob("/kaggle/input/*/*")):
    print(f"  {'[dir] ' if os.path.isdir(entry) else '[file]'} {entry}")

# ── EDIT THIS LINE only if auto-detect printed the wrong path ────────────────
DATA_ROOT = AUTO_ROOT

if DATA_ROOT is None:
    raise ValueError("Could not auto-detect FF++ root. Set DATA_ROOT manually above.")

REPO_URL   = "https://github.com/umardrazbhatti/EahnCode.git"
REPO_DIR   = "/kaggle/working/EahnCode"
OUTPUT_DIR = "/kaggle/working/outputs"
CACHE_DIR  = "/kaggle/working/.face_cache"

EPOCHS      = 10   # set 2 for a quick smoke-test; 10+ for real results
BATCH_SIZE  = 4
NUM_WORKERS = 0    # keep 0 on Kaggle to avoid CUDA/fork issues

print("\n" + "="*55)
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"REPO_DIR   : {REPO_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"EPOCHS     : {EPOCHS}")
print("="*55)
print("If DATA_ROOT looks wrong, edit this cell and re-run Cell 1 only.")


---
## Cell 2 — Sanity-check file counts
Both real and fake counts must be **> 0** before continuing.

In [ ]:
import glob, os

COMPRESSION   = "c23"
MANIPULATIONS = ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]

real_dir  = os.path.join(DATA_ROOT, "original_sequences", "youtube", COMPRESSION, "videos")
real_vids = glob.glob(os.path.join(real_dir, "*.mp4"))

print(f"Real videos directory : {real_dir}")
print(f"  exists : {os.path.isdir(real_dir)}")
print(f"  count  : {len(real_vids)}")
if real_vids:
    print(f"  sample : {os.path.basename(real_vids[0])}")

print()
total_fake = 0
for method in MANIPULATIONS:
    vdir  = os.path.join(DATA_ROOT, "manipulated_sequences", method, COMPRESSION, "videos")
    count = len(glob.glob(os.path.join(vdir, "*.mp4")))
    total_fake += count
    print(f"  [{'OK' if count > 0 else 'MISSING':7}]  {method:<18}  {count} videos")

print()
if len(real_vids) > 0 and total_fake > 0:
    print(f"READY — {len(real_vids)} real  |  {total_fake} fake  |  total {len(real_vids)+total_fake}")
else:
    print("ERROR — one or both classes are missing. Fix DATA_ROOT in Cell 1.")
    raise SystemExit("Stopping — fix DATA_ROOT first.")


---
## Cell 3 — Clean up previous run
Deletes old clone and outputs only. The dataset under `/kaggle/input/` is never touched.

In [ ]:
import shutil, os

def rm(path):
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"Removed dir : {path}")
    elif os.path.isfile(path):
        os.remove(path)
        print(f"Removed file: {path}")
    else:
        print(f"Not found (skip): {path}")

rm(REPO_DIR)
rm(OUTPUT_DIR)
rm(CACHE_DIR)
rm("/kaggle/working/eahn_outputs.zip")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("\nWorking directory after cleanup:")
print(os.listdir("/kaggle/working"))


---
## Cell 4 — Clone repo from GitHub
⚠️ **Prerequisite**: the corrected `data/datasets.py` must already be pushed to the repo before running this cell. The fix corrects the FF++ path layout and class imbalance.

In [ ]:
import os
os.system(f"git clone {REPO_URL} {REPO_DIR}")
print("\nRepo contents:")
print(os.listdir(REPO_DIR))


---
## Cell 5 — Install dependencies

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"],
    check=True
)

import importlib
print("\nKey package versions:")
for pkg in ["torch", "torchvision", "timm", "sklearn", "matplotlib", "seaborn", "cv2"]:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  OK      {pkg:<15} {ver}")
    except ImportError:
        print(f"  MISSING {pkg}")


---
## Cell 6 — Dataset loader verification ✅
Directly instantiates `DeepfakeDataset`, loads one batch and asserts both classes
are present. **Do not proceed to Cell 7 unless this prints `VERIFICATION PASSED`.**

If it prints `FAIL`, the corrected `data/datasets.py` has not been pushed to GitHub yet.
Re-push it, then re-run from Cell 3.


In [ ]:
import sys, torch
sys.path.insert(0, REPO_DIR)

from config import EAHNConfig
from data.datasets import DeepfakeDataset
from data.collate  import deepfake_collate_fn
from torch.utils.data import DataLoader

# Use minimal settings — this is a loader check only, not a training run
_cfg = EAHNConfig(
    data_root    = DATA_ROOT,
    output_dir   = OUTPUT_DIR,
    cache_dir    = CACHE_DIR,
    dataset_name = "ff++",
    num_frames   = 4,          # low for speed
    frame_size   = 224,
    train_split  = 0.8,
    val_split    = 0.1,
    device       = "auto",
    num_workers  = 0,
)

print("Building dataset...")
ds = DeepfakeDataset(_cfg, mode="train", dataset_type="ff++")
loader = DataLoader(ds, batch_size=8, collate_fn=deepfake_collate_fn, shuffle=True)

print("Loading one batch...")
batch = next(iter(loader))

labels_in_batch = batch["label"].unique().tolist()
frames_shape    = tuple(batch["frames"].shape)
has_mask_flags  = batch["has_mask"].tolist()

print(f"\n  Labels present  : {labels_in_batch}")
print(f"  Frames shape    : {frames_shape}")
print(f"  has_mask flags  : {has_mask_flags}")
print(f"  Mask shape      : {tuple(batch['mask'].shape)}")
print(f"  Total samples   : {len(ds)}")

# ── Assertions ────────────────────────────────────────────────────────────────
errors = []
if 0 not in labels_in_batch:
    errors.append("Real videos (label=0) missing from batch.")
if 1 not in labels_in_batch:
    errors.append("Fake videos (label=1) missing from batch.")
if frames_shape[1] != _cfg.num_frames:
    errors.append(f"Expected {_cfg.num_frames} frames per clip, got {frames_shape[1]}.")
if frames_shape[2:] != (3, 224, 224):
    errors.append(f"Unexpected frame tensor shape: {frames_shape[2:]}.")
if any(has_mask_flags):
    errors.append("has_mask should be False for this dataset (no masks available).")

if errors:
    print("\n" + "="*60)
    print("FAIL — Dataset verification did not pass:")
    for e in errors:
        print(f"  • {e}")
    print("="*60)
    print("Action: push the corrected data/datasets.py to GitHub,")
    print("then re-run from Cell 3.")
    raise SystemExit("Fix the errors above before training.")

print("\n" + "="*60)
print("VERIFICATION PASSED — both classes present, shapes correct.")
print("Proceed to Cell 7.")
print("="*60)


---
## Cell 7 — Train + Evaluate
Runs the full pipeline: training → evaluation → graphs → heatmap videos → explanations.
Training progress is printed per epoch. Expect ~4–6 min per epoch on a T4 GPU.


In [ ]:
import os
os.chdir(REPO_DIR)

cmd = (
    f"python run_full_pipeline.py"
    f" --data_root           \"{DATA_ROOT}\""
    f" --dataset_name        ff++"
    f" --dataset_compression c23"
    f" --output_dir          \"{OUTPUT_DIR}\""
    f" --cache_dir           \"{CACHE_DIR}\""
    f" --epochs              {EPOCHS}"
    f" --batch_size          {BATCH_SIZE}"
    f" --num_workers         {NUM_WORKERS}"
    f" --eval_after_train"
)
print("Running:", cmd)
os.system(cmd)


---
## Cell 8 — Check output files produced

In [ ]:
import os

EXPECTED = [
    "metrics.csv",
    "roc_curve.png",
    "pr_curve.png",
    "confusion_matrix.png",
    "score_distribution.png",
    "best_model.pth",
]

print(f"Output directory: {OUTPUT_DIR}\n")
all_ok = True
for fname in EXPECTED:
    fpath  = os.path.join(OUTPUT_DIR, fname)
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) if exists else 0
    status = "OK     " if exists else "MISSING"
    size_s = f"({size:,} bytes)" if exists else ""
    print(f"  {status}  {fname}  {size_s}")
    if not exists:
        all_ok = False

for subdir in ["heatmaps", "explanations"]:
    d = os.path.join(OUTPUT_DIR, subdir)
    if os.path.isdir(d):
        files = os.listdir(d)
        print(f"\n  {subdir}/  — {len(files)} file(s)")
        for f in sorted(files)[:5]:
            print(f"    {f}")
        if len(files) > 5:
            print(f"    ... and {len(files)-5} more")
    else:
        print(f"\n  {subdir}/  MISSING")

print("\n" + ("All expected outputs present." if all_ok else "WARNING: some outputs missing — check Cell 7 for errors."))


---
## Cell 9 — Display detection metrics table

In [ ]:
import csv, os
import pandas as pd
from IPython.display import display

csv_path = os.path.join(OUTPUT_DIR, "metrics.csv")
if not os.path.exists(csv_path):
    print("metrics.csv not found. Did Cell 7 complete successfully?")
else:
    rows = []
    with open(csv_path, newline="") as f:
        reader = csv.reader(f)
        next(reader, None)          # skip header
        for row in reader:
            if len(row) == 2:
                try:
                    rows.append({"Metric": row[0], "Value": f"{float(row[1]):.4f}"})
                except ValueError:
                    rows.append({"Metric": row[0], "Value": row[1]})

    print("="*45)
    print("  EAHN Evaluation Metrics")
    print("="*45)
    display(pd.DataFrame(rows).style.set_properties(**{"text-align": "left"}))


---
## Cell 10 — Display detection graphs

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

graphs = [
    ("roc_curve.png",          "ROC Curve"),
    ("pr_curve.png",           "Precision-Recall Curve"),
    ("confusion_matrix.png",   "Confusion Matrix"),
    ("score_distribution.png", "Score Distribution"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (fname, title) in zip(axes.flatten(), graphs):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        ax.imshow(mpimg.imread(fpath))
        ax.set_title(title, fontsize=13, fontweight="bold")
    else:
        ax.text(0.5, 0.5, f"{fname}\nnot found", ha="center", va="center", color="red")
    ax.axis("off")
plt.tight_layout()
plt.show()


---
## Cell 11 — Display heatmap videos (first sample, all 4 XAI methods)

In [ ]:
from IPython.display import Video, HTML, display
import os

heatmap_dir = os.path.join(OUTPUT_DIR, "heatmaps")
if not os.path.isdir(heatmap_dir):
    print("No heatmaps directory found.")
else:
    methods   = ["intrinsic", "gradcam", "rollout", "shap"]
    mp4_files = sorted(f for f in os.listdir(heatmap_dir) if f.endswith(".mp4"))
    video_ids = sorted(set(
        f.replace(f"_{m}.mp4", "")
        for f in mp4_files
        for m in methods
        if f.endswith(f"_{m}.mp4")
    ))

    if not video_ids:
        print("No heatmap videos found in", heatmap_dir)
    else:
        vid = video_ids[0]
        display(HTML(f"<h3>Sample: {vid}</h3>"))
        for method in methods:
            fpath = os.path.join(heatmap_dir, f"{vid}_{method}.mp4")
            if os.path.exists(fpath):
                display(HTML(f"<b>{method.upper()}</b>"))
                display(Video(fpath, embed=True, width=480))
            else:
                print(f"  [{method}] not found")
        if len(video_ids) > 1:
            print(f"\n({len(video_ids)-1} more sample(s) available in {heatmap_dir})")


---
## Cell 12 — Print plain-English explanations

In [ ]:
import os

exp_dir = os.path.join(OUTPUT_DIR, "explanations")
if not os.path.isdir(exp_dir):
    print("No explanations directory found.")
else:
    txt_files = sorted(f for f in os.listdir(exp_dir) if f.endswith(".txt"))
    if not txt_files:
        print("No .txt explanation files found.")
    else:
        for fname in txt_files[:3]:
            print("\n" + "="*60)
            print(f" {fname}")
            print("="*60)
            with open(os.path.join(exp_dir, fname), encoding="utf-8") as f:
                print(f.read())
        if len(txt_files) > 3:
            print(f"\n... and {len(txt_files)-3} more in {exp_dir}")


---
## Cell 13 — Full dashboard summary

In [ ]:
import sys, os
sys.path.insert(0, REPO_DIR)
from scripts.dashboard import show_dashboard
show_dashboard(OUTPUT_DIR)


---
## Cell 14 — Zip outputs for download
Download `eahn_outputs.zip` from the Kaggle **Output** tab after this cell completes.

In [ ]:
import shutil, os

zip_base = "/kaggle/working/eahn_outputs"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
full_zip = zip_base + ".zip"
size_mb  = os.path.getsize(full_zip) / 1e6
print(f"Created  : {full_zip}")
print(f"Size     : {size_mb:.1f} MB")
print("Download via Kaggle Output tab → eahn_outputs.zip")
